In [ ]:
import rasterio
import numpy as np
import pandas as pd


def preprocess_single_band_soil_raster(input_raster, output_raster, soil_meta, strict=False):
    """
    Preprocess a single-band soil taxonomy raster:
    - Map soil taxonomy codes (from Excel) → soil orders
    - Soil orders → numeric IDs
    - Unknown/unmapped codes → 0
    - Handle nodata values (65535, 117, 157)
    """

    # Soil order suffix codes and names
    code = ['alfs', 'ands', 'els', 'ents',
            'epts', 'erts', 'ids', 'ists',
            'ods', 'olls', 'oxs', 'ults']

    soil = ['Alfisol', 'Andisol', 'Gelisol', 'Entisol',
            'Inceptisol', 'Vertisol', 'Aridisol', 'Histosol',
            'Spodosol', 'Mollisol', 'Oxisol', 'Ultisol']

    # Add soil order column in metadata
    soil_meta['soil'] = ''
    for i in range(len(soil_meta)):
        character = soil_meta.loc[i, "soil_taxonomy1"]
        for n in range(len(code)):
            if str(character).endswith(code[n]):
                soil_meta.loc[i, 'soil'] = soil[n]

    # Dictionary: soil taxonomy code → soil order name
    soil_mergable = soil_meta.set_index("soil_taxonomy")["soil"].to_dict()

    # Map soil names → numeric IDs
    soil_to_num = {s: i + 1 for i, s in enumerate(soil)}
    soil_to_num["Unknown"] = 0
    nodata_val = 65535  # output NoData value

    # Open raster
    with rasterio.open(input_raster) as src:
        band = src.read(1)
        meta = src.meta.copy()

    # Define nodata values
    nodata_vals = {65535, 117, 157}

    # Original unique values
    unique_vals, counts = np.unique(band, return_counts=True)
    print("\n📊 Original raster unique values:")
    for v, c in zip(unique_vals, counts):
        print(f"  {v}: {c}")

    # Report nodata counts
    for nv in nodata_vals:
        if nv in unique_vals:
            print(f"  ⚠️ Found {counts[list(unique_vals).index(nv)]} pixels with value {nv} (will be set as NoData)")

    # Check for unmapped values (excluding nodata-like values)
    all_raster_vals = set(unique_vals) - nodata_vals
    valid_vals = set(soil_mergable.keys())
    unmapped = sorted(all_raster_vals - valid_vals)

    if unmapped:
        if strict:
            raise ValueError(f"❌ Unmapped raster values found (excluding nodata-like values): {unmapped}")
        else:
            print(f"⚠️ Warning: Unmapped values {unmapped} -> assigned to 0 (Unknown)")

    # Map raster values directly to numeric IDs
    def map_to_numeric(x):
        if x in nodata_vals:
            return nodata_val
        soil_name = soil_mergable.get(x, "Unknown")
        return soil_to_num.get(soil_name, 0)

    soil_band_num = np.vectorize(map_to_numeric)(band).astype(np.int32)

    # New unique values
    unique_vals, counts = np.unique(soil_band_num, return_counts=True)
    print("\n📊 New raster unique values (numeric classes):")
    for v, c in zip(unique_vals, counts):
        if v == nodata_val:
            label = "NoData"
        else:
            label = [k for k, vv in soil_to_num.items() if vv == v]
            label = label[0] if label else "Unknown"
        print(f"  {v} ({label}): {c}")

    # Save new raster
    meta.update(count=1, dtype=rasterio.int32, nodata=nodata_val)
    with rasterio.open(output_raster, "w", **meta) as dst:
        dst.write(soil_band_num, 1)

    print(f"\n✅ Preprocessed soil taxonomy raster saved to {output_raster}")

    #print percentage of NoData pixels
    total_pixels = soil_band_num.size
    nodata_pixels = np.sum(soil_band_num == nodata_val)
    nodata_percentage = (nodata_pixels / total_pixels) * 100
    print(f"📉 NoData pixels: {nodata_pixels} ({nodata_percentage:.2f}%)")


# Example usage
if __name__ == "__main__":
    raster_stack_file = "GeoFiles/Soil/Reprojected_Soil_Taxonomy.tif"
    output_file = "GeoFiles/Soil/AZ_Soil_Taxonomy_Preprocessed.tif"
    soil_meta = pd.read_excel("GeoFiles/Soil/LU_soil.xlsx", sheet_name="soil")

    preprocess_single_band_soil_raster(raster_stack_file, output_file, soil_meta, strict=False)
